# Render JPG → MP4 trên Google Drive (Colab, chạy tiếp được khi bị ngắt)

**Cách dùng**
1. `Runtime → Change runtime type → T4 GPU` (khuyến nghị — encode bằng NVENC nhanh hơn CPU nhiều lần; không có GPU vẫn chạy bằng CPU).
2. Sửa ô **Cấu hình**, rồi `Runtime → Run all`.
3. **Nếu bị ngắt kết nối:** kết nối lại và `Run all` lần nữa. Video đã xong sẽ được bỏ qua, chỉ mất video đang làm dở.

**Cách hoạt động (từng video một):**
copy frame JPG từ Drive về ổ local `/content` (song song nhiều luồng; copy video kế tiếp trong lúc encode video hiện tại) → encode bằng FFmpeg → kiểm tra số frame trong MP4 = số JPG → copy MP4 lên Drive dưới tên `.tmp.mp4` rồi đổi tên.
Vì vậy trên Drive **không bao giờ có file MP4 dở dang** mang tên chính thức.

In [ ]:
# 1) Mount Drive + kiểm tra môi trường
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi -L || echo "Không có GPU -> sẽ encode bằng CPU (libx264)"
!ffmpeg -hide_banner -version | head -1
!df -h /content | tail -1

In [ ]:
# 2) CẤU HÌNH — sửa ở đây
ROOT_DIR = "/content/drive/MyDrive/Harmony4D"   # thư mục gốc dữ liệu trên Drive

# "harmony4d": quét đệ quy ROOT_DIR tìm <tên_sequence>/exo/camX/images
# "list"     : chỉ render các thư mục trong IMAGE_DIRS
MODE = "harmony4d"
IMAGE_DIRS = [
    # "/content/drive/MyDrive/abc/cam1/images",
]
SKIP_CAM0 = True          # giống script cũ: bỏ qua cam0

# None = ghi MP4 cạnh dữ liệu (giống script cũ):
#   harmony4d: <...>/camX/images/harmony4D_<sequence>_camX_ffmpeg.mp4
#   list     : <thư_mục_cha>/<tên_thư_mục_cha>.mp4
# Hoặc đặt 1 thư mục trên Drive để gom tất cả video vào đó.
OUTPUT_DIR = None

VIDEO_PREFIX = "harmony4D"
FPS = 20
ENCODER = "auto"          # "auto" (NVENC nếu có, lỗi thì lùi về x264) | "nvenc" | "x264"
QUALITY = 23              # CQ/CRF: nhỏ hơn = nét hơn, file to hơn (18–28 là hợp lý)
X264_PRESET = "veryfast"  # chỉ dùng khi encode bằng CPU
COPY_WORKERS = 16         # số luồng copy JPG từ Drive
LOCAL_WORK = "/content/render_work"
LOG_PATH = None           # None = <ROOT_DIR>/_render_log.jsonl (nhật ký video đã xong)

In [ ]:
# 3) Hàm tiện ích
import os, re, json, time, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from tqdm.auto import tqdm

JPG_RE = re.compile(r"^(.*?)(\d+)(\.jpe?g)$", re.IGNORECASE)
LOG_FILE = LOG_PATH or os.path.join(ROOT_DIR, "_render_log.jsonl")


def find_jpg_sequence(images_dir):
    # Trả về (danh sách file theo thứ tự, pattern ffmpeg, số bắt đầu)
    parsed = []
    for name in os.listdir(images_dir):
        m = JPG_RE.match(name)
        if m:
            prefix, num, suffix = m.groups()
            parsed.append((int(num), len(num), prefix, suffix))
    if not parsed:
        raise ValueError("không có file JPG")
    if len({(w, p, s) for _, w, p, s in parsed}) != 1:
        raise ValueError("tên frame không đồng nhất (prefix / đuôi / độ rộng số)")
    parsed.sort()
    nums = [x[0] for x in parsed]
    if nums != list(range(nums[0], nums[0] + len(nums))):
        missing = sorted(set(range(nums[0], nums[-1] + 1)) - set(nums))
        raise ValueError(f"thiếu frame: {missing[:10]}{' ...' if len(missing) > 10 else ''}")
    _, w, p, s = parsed[0]
    files = [f"{p}{n:0{w}d}{s}" for n in nums]
    pattern = f"{p.replace('%', '%%')}%0{w}d{s.replace('%', '%%')}"
    return files, pattern, nums[0]


def scan_tasks():
    tasks = []
    if MODE == "list":
        for d in IMAGE_DIRS:
            d = os.path.abspath(d)
            parent = os.path.dirname(d)
            pname = os.path.basename(parent) or "video"
            tasks.append({"images_dir": d,
                          "output": os.path.join(OUTPUT_DIR or parent, f"{pname}.mp4")})
    elif MODE == "harmony4d":
        print("Đang quét", ROOT_DIR, "...")
        for cur, dirnames, _ in os.walk(ROOT_DIR):
            if os.path.basename(cur) != "exo":
                # không liệt kê các thư mục ảnh nặng không nằm trong exo
                dirnames[:] = [x for x in dirnames if x != "images" and not x.startswith(".")]
                continue
            seq = os.path.basename(os.path.dirname(cur))
            for cam in sorted(os.listdir(cur)):
                m = re.fullmatch(r"cam(\d+)", cam)
                if not m or (SKIP_CAM0 and int(m.group(1)) == 0):
                    continue
                img = os.path.join(cur, cam, "images")
                if os.path.isdir(img):
                    name = f"{VIDEO_PREFIX}_{seq}_{cam}_ffmpeg.mp4"
                    tasks.append({"images_dir": img,
                                  "output": os.path.join(OUTPUT_DIR or img, name)})
            dirnames.clear()
    else:
        raise ValueError("MODE phải là 'harmony4d' hoặc 'list'")

    outs = [t["output"] for t in tasks]
    dup = {o for o in outs if outs.count(o) > 1}
    if dup:
        raise ValueError(f"Nhiều task trùng file đầu ra (đổi OUTPUT_DIR=None?): {sorted(dup)[:3]}")

    good = []
    for t in tqdm(tasks, desc="Đọc danh sách frame"):
        try:
            files, pattern, start = find_jpg_sequence(t["images_dir"])
        except (ValueError, OSError) as e:
            print(f"[Bỏ qua] {t['images_dir']}: {e}")
            continue
        t.update(files=files, pattern=pattern, start=start, n=len(files))
        good.append(t)
    return good


def count_frames(path):
    # Số frame (packet) của luồng video; -1 nếu đọc lỗi
    try:
        r = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_packets",
             "-show_entries", "stream=nb_read_packets", "-of", "json", path],
            capture_output=True, text=True, timeout=900, check=True)
        return int(json.loads(r.stdout)["streams"][0]["nb_read_packets"])
    except Exception:
        return -1


def load_log():
    done = {}
    if os.path.isfile(LOG_FILE):
        with open(LOG_FILE, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    done[rec["output"]] = rec
                except Exception:
                    pass
    return done


def append_log(rec):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()


def is_done(task, log):
    out = task["output"]
    if not os.path.isfile(out) or os.path.getsize(out) == 0:
        return False
    rec = log.get(out)
    # Đường nhanh: nhật ký khớp số frame và kích thước file -> không cần đọc lại video từ Drive
    if rec and rec.get("frames") == task["n"] and rec.get("size") == os.path.getsize(out):
        return True
    return count_frames(out) == task["n"]


def nvenc_available():
    r = subprocess.run(
        ["ffmpeg", "-hide_banner", "-v", "error", "-f", "lavfi",
         "-i", "testsrc2=size=1280x720:rate=20", "-frames:v", "10",
         "-c:v", "h264_nvenc", "-f", "null", "-"],
        capture_output=True)
    return r.returncode == 0


def fetch_local(task, idx):
    # Copy song song toàn bộ frame của 1 task từ Drive về ổ local
    local_dir = os.path.join(LOCAL_WORK, f"task_{idx}")
    shutil.rmtree(local_dir, ignore_errors=True)
    os.makedirs(local_dir)
    t0 = time.time()

    def cp(name):
        shutil.copyfile(os.path.join(task["images_dir"], name), os.path.join(local_dir, name))

    with ThreadPoolExecutor(COPY_WORKERS) as ex:
        list(ex.map(cp, task["files"]))
    return local_dir, time.time() - t0


def encode_cmd(local_dir, task, out, enc):
    cmd = ["ffmpeg", "-y", "-hide_banner", "-v", "error",
           "-framerate", str(FPS), "-start_number", str(task["start"]),
           "-i", os.path.join(local_dir, task["pattern"]),
           # ép kích thước chẵn (yuv420p yêu cầu), không đổi gì nếu đã chẵn
           "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", "-pix_fmt", "yuv420p"]
    if enc == "nvenc":
        cmd += ["-c:v", "h264_nvenc", "-preset", "p4", "-rc", "vbr", "-cq", str(QUALITY), "-b:v", "0"]
    else:
        cmd += ["-c:v", "libx264", "-preset", X264_PRESET, "-crf", str(QUALITY)]
    return cmd + ["-movflags", "+faststart", out]


def render_one(task, local_dir, encoders):
    local_out = local_dir + ".mp4"
    err = ""
    t0 = time.time()
    for enc in encoders:
        r = subprocess.run(encode_cmd(local_dir, task, local_out, enc), capture_output=True, text=True)
        if r.returncode == 0:
            got = count_frames(local_out)
            if got == task["n"]:
                break
            err = f"[{enc}] MP4 có {got} frame, cần {task['n']}"
        else:
            err = f"[{enc}] {r.stderr.strip()[-800:]}"
        if enc != encoders[-1]:
            print(f"  {err} -> thử encoder khác")
    else:
        raise RuntimeError(err)
    enc_time = time.time() - t0

    # Upload: ghi .tmp.mp4 rồi đổi tên -> không để lại file chính thức bị dở
    t1 = time.time()
    out = task["output"]
    os.makedirs(os.path.dirname(out), exist_ok=True)
    tmp = out[:-4] + ".tmp.mp4"
    shutil.copyfile(local_out, tmp)
    os.replace(tmp, out)
    size = os.path.getsize(out)
    os.remove(local_out)
    return enc, enc_time, time.time() - t1, size

print("OK")

In [ ]:
# 4) Quét dữ liệu và xác định video còn phải làm
if ENCODER == "x264":
    ENCODERS = ["x264"]
elif nvenc_available():
    ENCODERS = ["nvenc"] if ENCODER == "nvenc" else ["nvenc", "x264"]
else:
    if ENCODER == "nvenc":
        raise SystemExit("ENCODER='nvenc' nhưng NVENC không dùng được (chưa bật GPU?)")
    ENCODERS = ["x264"]
print("Encoder:", " -> ".join(ENCODERS))

tasks = scan_tasks()
log = load_log()
pending, done = [], []
for t in tqdm(tasks, desc="Kiểm tra video đã có"):
    (done if is_done(t, log) else pending).append(t)

total_frames = sum(t["n"] for t in pending)
print(f"\nTổng task: {len(tasks)} | đã xong: {len(done)} | cần render: {len(pending)} ({total_frames:,} frame)")
for t in pending[:20]:
    print(f"  - {t['n']:>6} frame  {t['output']}")
if len(pending) > 20:
    print(f"  ... và {len(pending) - 20} task nữa")

In [ ]:
# 5) RENDER — chạy lại ô này (hoặc Run all) sau khi bị ngắt: video đã xong được bỏ qua
os.makedirs(LOCAL_WORK, exist_ok=True)
failed = []
frames_done, t_start = 0, time.time()

with ThreadPoolExecutor(1) as prefetch, tqdm(total=total_frames, unit="frame", desc="Render") as bar:
    fut = prefetch.submit(fetch_local, pending[0], 0) if pending else None
    for i, task in enumerate(pending):
        local_dir = None
        try:
            local_dir, copy_time = fut.result()
        except Exception as e:
            failed.append((task["output"], f"copy lỗi: {e}"))
            print(f"\n[Lỗi copy] {task['images_dir']}: {e}")
        # bắt đầu copy task kế tiếp trong lúc encode task này
        fut = prefetch.submit(fetch_local, pending[i + 1], i + 1) if i + 1 < len(pending) else None

        if local_dir:
            try:
                enc, enc_time, up_time, size = render_one(task, local_dir, ENCODERS)
                append_log({"output": task["output"], "frames": task["n"], "size": size,
                            "encoder": enc, "fps": FPS, "time": datetime.now().isoformat(timespec="seconds")})
                tqdm.write(f"[{i + 1}/{len(pending)}] {os.path.basename(task['output'])}: {task['n']} frame | "
                           f"copy {copy_time:.0f}s | encode {enc_time:.0f}s ({task['n'] / max(enc_time, 1e-6):.0f} fps, {enc}) | "
                           f"upload {up_time:.0f}s | {size / 1e6:.0f} MB")
            except Exception as e:
                failed.append((task["output"], str(e)))
                tqdm.write(f"[Lỗi] {task['output']}\n  {e}")
            finally:
                shutil.rmtree(local_dir, ignore_errors=True)

        frames_done += task["n"]
        bar.update(task["n"])
        rate = frames_done / (time.time() - t_start)
        eta = datetime.now() + timedelta(seconds=(total_frames - frames_done) / max(rate, 1e-6))
        bar.set_postfix_str(f"{rate:.0f} frame/s tổng thể | xong lúc ~{eta:%H:%M}")

print(f"\nHoàn tất: {len(pending) - len(failed)} thành công, {len(failed)} lỗi, "
      f"tổng thời gian {(time.time() - t_start) / 60:.1f} phút")
for out, reason in failed:
    print(f"- {out}\n  {reason}")
if failed:
    print("Chạy lại ô 4 và 5 để thử lại các video lỗi.")

In [ ]:
# 6) (Tuỳ chọn) Chạy khi đã xong hết: đẩy nốt dữ liệu đang chờ đồng bộ lên Drive rồi tháo Drive
drive.flush_and_unmount()
print("Đã đồng bộ xong lên Drive.")